# **Agrupamento de Leituras RSSI por Roteador**

---

Versão 1.0.0

## **Importação das Bibliotecas**

In [1]:
import pandas as pd

## **Configuração e Leitura do Arquivo CSV**

In [8]:
# ===== ARQUIVOS =====
arquivo_entrada = "../data/source/FS220426EE10.csv"
arquivo_saida = "../data/signs/AgrupamentoDeRedes3.csv"

# ===== LEITURA =====
df = pd.read_csv(arquivo_entrada)

# ===== LIMPEZA =====
df["ambiente"] = df["ambiente"].str.strip()
df["bssid"] = df["bssid"].str.lower()

# ===== GARANTIR ORDEM ORIGINAL =====
#df = df.reset_index(drop=True)

## **Agrupamento de Leituras**

In [9]:
# ===== CRIAR CHAVE DO ROTEADOR =====
# Remove último byte do MAC + canal
df["base_mac"] = df["bssid"].str[:-2]
df["chave_roteador"] = df["base_mac"] + "_" + df["canal"].astype(str)

# ===== AGRUPAMENTO =====
df_saida = (
    df.groupby(["ambiente", "bssid", "canal", "chave_roteador"])
    .agg(
        Minimo=("rssi", "min"),
        Maximo=("rssi", "max"),
        Quantidade=("rssi", "count")
    )
    .reset_index()
)

# ===== ORDENAR (ANTES DE NUMERAR!) =====
df_saida = df_saida.sort_values(by=["ambiente", "chave_roteador"])

# ===== NUMERAR POR ORDEM DE APARIÇÃO =====
mapa_roteadores = {}
contador = 1

for chave in df_saida["chave_roteador"]:
    if chave not in mapa_roteadores:
        mapa_roteadores[chave] = f"Roteador {str(contador).zfill(2)}"
        contador += 1

df_saida["NumRoteador"] = df_saida["chave_roteador"].map(mapa_roteadores)

# ===== RENOMEAR COLUNAS =====
df_saida = df_saida.rename(columns={
    "ambiente": "Local",
    "bssid": "Roteador",
    "canal": "Canal"
})

# ===== ORDENAR FINAL (AGORA COM NUMERO CERTO) =====
df_saida = df_saida.sort_values(by=["Local", "NumRoteador", "Canal"])

# ===== REMOVER COLUNA AUXILIAR =====
df_saida = df_saida.drop(columns=["chave_roteador"])

# ===== SALVAR =====
df_saida.to_csv(arquivo_saida, index=False)

print("Arquivo gerado com sucesso!")

Arquivo gerado com sucesso!
